In [6]:
from IPython.display import Audio, display
import scipy.io.wavfile as wavfile
import sounddevice as sd


duration = 3
samplerate = 16_000

audio = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1)
sd.wait()
audio = audio.flatten()

display(Audio(audio, rate=samplerate))

In [ ]:
from dotenv import load_dotenv
from google.genai import types, Client
import os
import io

load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")

client = Client(api_key=API_KEY)

system_prompt = """
You're a helpful assistant
"""

buffer = io.BytesIO()
wavfile.write(buffer, samplerate, audio)
buffer.seek(0)

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
        types.Part.from_bytes(
            data=buffer.read(),
            mime_type="audio/wav"
        ),
        "Analyze the user's speech and respond according to your instructions."
    ],
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
    )
)

print(f"{response.text}")

A bochechuda que não toma banho.


In [13]:
import edge_tts

text = response.text
voice = "pt-BR-AntonioNeural"

communicate =  edge_tts.Communicate(text, voice)

audio_buffer = io.BytesIO()

async for chunk in communicate.stream():
    if chunk["type"] == "audio":
        audio_buffer.write(chunk["data"])

audio_buffer.seek(0)
display(Audio(audio_buffer.read(), autoplay=True))